In [1]:
import pandas as pd
import numpy as np

# Загружаем подготовленные данные
df = pd.read_csv("../data/processed/demand_real.csv", parse_dates=["date"])
df = df.sort_values(["product", "date"]).reset_index(drop=True)
df.head()

,date,product,demand_tons,month
0,2013-01-01,Item_1,328,1
1,2013-02-01,Item_1,322,2
2,2013-03-01,Item_1,477,3
3,2013-04-01,Item_1,522,4
4,2013-05-01,Item_1,531,5


Генерация лагов и скользящих средних:
В этой ячейке мы применим функцию к каждой группе товаров отдельно, чтобы значения не перемешивались между разными продуктами.

In [ ]:
def build_features(group):
    g = group.copy()
    
    # 1. Лаги (значения в прошлые периоды)
    for lag in [1, 2, 3, 12]:
        g[f"lag_{lag}"] = g["demand_tons"].shift(lag)
        
    # 2. Скользящие статистики (среднее и стандартное отклонение за последние 3 месяца)
    g["rolling_mean_3"] = g["demand_tons"].shift(1).rolling(window=3).mean()
    g["rolling_std_3"]  = g["demand_tons"].shift(1).rolling(window=3).std()
    
    # 3. Календарные признаки
    g["month"]   = g["date"].dt.month
    g["quarter"] = g["date"].dt.quarter
    g["year"]    = g["date"].dt.year
    
    return g

df_feat = df.groupby("product", group_keys=False).apply(build_features)

df_feat = df_feat.dropna().reset_index(drop=True)

print(f"Размер данных после создания признаков: {df_feat.shape}")
df_feat.head()

Размер данных после создания признаков: (240, 11)


,date,demand_tons,month,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_std_3,quarter,year
0,2014-01-01,435,1,388.0,572.0,484.0,328.0,481.333333,92.028981,1,2014
1,2014-02-01,405,2,435.0,388.0,572.0,322.0,465.000000,95.598117,1,2014
2,2014-03-01,549,3,405.0,435.0,388.0,477.0,409.333333,23.797759,1,2014
3,2014-04-01,573,4,549.0,405.0,435.0,522.0,463.000000,75.973680,2,2014
4,2014-05-01,651,5,573.0,549.0,405.0,531.0,509.000000,90.862534,2,2014


Хронологическое разделение на Train и Test:
Для временных рядов нельзя использовать случайное разделение (train_test_split), так как мы не можем предсказывать прошлое, используя будущее. Мы возьмем последние 12 месяцев (2017 год) для теста, а всё, что до этого — для обучения.

In [3]:
# Определяем дату отсечения (последний год оставляем на тест)
cutoff_date = pd.to_datetime("2017-01-01")

train = df_feat[df_feat["date"] < cutoff_date].copy()
test  = df_feat[df_feat["date"] >= cutoff_date].copy()

print(f"Train: {train['date'].min().date()} — {train['date'].max().date()}, строк: {len(train)}")
print(f"Test:  {test['date'].min().date()} — {test['date'].max().date()}, строк: {len(test)}")

Train: 2014-01-01 — 2016-12-01, строк: 180
Test:  2017-01-01 — 2017-12-01, строк: 60


Сохранение подготовленных датасетов:

In [4]:
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv",  index=False)
print("Train и Test сохранены в папку data/processed/")

Train и Test сохранены в папку data/processed/
